In [33]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import utils.base_utils as bu
import utils.window_utils as wu
from utils.macro_grouping import add_group_level, build_full_group_mapping, groups_as_array

from models.base import *
from models.classical import *
from models.ann_vector_validation import *
from models.linear import *
from models.tree import *

## Bianchi (1971-2018): Forward-only, revised data

In [34]:
start_date = '1971-08-31'
end_date = '2018-12-31'
OOS_start = pd.Timestamp('1990-01-31')

yield_type = 'lw'
use_macro = False   # False now, easy to switch later
seeds = [1]

target_cols = ['24', '36', '48', '60', '84', '120']

maturities = [str(i) for i in range(12, 121) if i % 12 == 0]

yields = bu.get_yields(type=yield_type, start=start_date, end=end_date, maturities=maturities)
forward = bu.get_forward_rates(yields)
xr = bu.get_excess_returns(yields, horizon=12).dropna()

fred_md = None

if use_macro:
    fred_md_raw = bu.get_fred_data('data/2026-01-MD.csv', start=start_date, end=end_date)

In [35]:
yields = yields.loc[yields.index <= xr.index[-1]]
forward = forward.loc[forward.index <= xr.index[-1]]
xr = xr.loc[xr.index <= xr.index[-1]]

if fred_md is not None:
    fred_md = fred_md.loc[fred_md.index <= xr.index[-1]]


In [36]:
if use_macro:
    X = pd.concat([fred_md, forward, yields], axis=1, keys=['fred', 'forward', 'yields'])
    s2g = build_full_group_mapping(fred_md, forward, yields)
    X = add_group_level(X, s2g, level_name='group')
    X = X.sort_index(axis=1, level='group')
    groups = groups_as_array(X, level='group')
else:
    X = pd.concat([forward, yields], axis=1, keys=['forward', 'yields'])
    groups = None

dates = xr.index

In [37]:
targets = {
    '24': xr['24'],
    '36': xr['36'],
    '48': xr['48'],
    '60': xr['60'],
    '84': xr['84'],
    '120': xr['120'],
}

In [6]:
models = {
    #'PCA(3)': PCABaselineModel(components=3, series='forward'),
    #'PCA(5)': PCABaselineModel(components=5, series='forward'),
    #'PCA(10)': PCABaselineModel(components=10, series='forward'),

    #'Ridge': RidgeModel(series='forward'),
    #'Lasso': LassoModel(series='forward'),
    #'BianchiENet': BianchiElasticNet(),
    #'ElasticNet': ElasticNetModel(series='forward'),

    #'ExtraTrees': ExtraTreesModel(features={'forward': {'method': 'raw'},}),
    'RandomForest': RandomForestForwardModel(),
    # 'GBRT': GradientBoostingModel(features={'forward': {'method': 'raw'}}),

    # 'NN_1x3': NeuralNetModel(
    #     features={'forward': {'method': 'raw'}},
    #     hidden_layers=(3,)
    # ),
}

In [ ]:
results = []
forecast_store = {}

In [ ]:
for seed in seeds:
    print(f'Running seed {seed}...')

    for model_name, model in models.items():
        print(f'\nModel: {model_name}')

        for target_name, y in targets.items():
            print(f'  Target: {target_name}')

            y_forecast = wu.expanding_window(
                model,
                X,
                y,
                dates,
                OOS_start,
                gap=0,
                refit_freq=1,
                coef_callback=None
            )

            # store results
            forecast_store.setdefault(target_name, {})
            forecast_store[target_name][model_name] = y_forecast

            # compute metrics
            r2_hist = wu.oos_r2(y, y_forecast, benchmark='hist_mean')
            pval_hist = bu.RSZ_Signif(y, y_forecast)

            results.append({
                'seed': seed,
                'target': target_name,
                'model': model_name,
                'r2_hist_mean': r2_hist,
                'pval_hist_mean': pval_hist
            })

In [ ]:
results_df = pd.DataFrame(results)

results_df_display = results_df.copy()

results_df_display['pval_hist_mean'] = results_df_display.apply(
    lambda row: row['pval_hist_mean'] if row['r2_hist_mean'] > 0 else np.nan,
    axis=1
)

results_df_display['r2_hist_mean'] = results_df_display['r2_hist_mean'] * 100

results_df_display = results_df_display.round({
    'r2_hist_mean': 1,
    'pval_hist_mean': 4
})

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
print(results_df_display)


## Bianchi (1971-2018): Forward + macro, revised data

In [38]:
start_date = '1971-08-31'
end_date = '2018-12-31'
OOS_start = pd.Timestamp('1990-01-31')

use_macro = True
seeds = [1]
maturities = [str(i) for i in range(12, 121) if i % 12 == 0]

target_cols = ['24', '36', '48', '60', '84', '120']

yields = bu.get_yields(type='lw', start=start_date, end=end_date, maturities=maturities)
forward = bu.get_forward_rates(yields)
xr = bu.get_excess_returns(yields, horizon=12).dropna()

fred_md = None
fred_md_start_date = pd.to_datetime(start_date) - pd.DateOffset(months=6)
if use_macro:
    fred_md = bu.get_fred_data('data/2026-01-MD.csv', start=fred_md_start_date, end=end_date)

fred_md = fred_md.shift(1) 


fred_md = fred_md[start_date:end_date]
fred_md = fred_md.drop(columns=['ACOGNO', 'TWEXAFEGSMTHx', 'UMCSENTx'])
# Align all predictors to the target sample end date
yields = yields.loc[yields.index <= xr.index[-1]]
forward = forward.loc[forward.index <= xr.index[-1]]
if fred_md is not None:
    fred_md = fred_md.loc[fred_md.index <= xr.index[-1]]

# Build predictor matrix
if use_macro:
    X = pd.concat([fred_md, forward, yields], axis=1, keys=['macro', 'forward', 'yields'])
else:
    X = pd.concat([forward, yields], axis=1, keys=['forward', 'yields'])

groups = None
dates = xr.index

In [39]:
print(fred_md.head())

                 RPI   W875RX1  DPCERA3M086SBEA  CMRMTSPLx   RETAILx  \
date                                                                   
1971-08-31  0.003416  0.003781         0.005686   0.006265  0.005563   
1971-09-30  0.005878  0.004373         0.009921   0.008964  0.017103   
1971-10-31  0.002766  0.004064         0.002436   0.000692  0.012000   
1971-11-30  0.005072  0.004144         0.005047   0.016765  0.011525   
1971-12-31  0.005221  0.006064         0.006435   0.009294 -0.000638   

              INDPRO   IPFPNSS   IPFINAL   IPCONGD  IPDCONGD  IPNCONGD  \
date                                                                     
1971-08-31 -0.005822 -0.005807 -0.003355 -0.011511 -0.009855 -0.012203   
1971-09-30  0.016122  0.014228  0.010855  0.009174 -0.001675  0.013913   
1971-10-31  0.007459  0.010270  0.010405  0.013436  0.017961  0.011509   
1971-11-30  0.004233  0.005784  0.005357  0.007925  0.009920  0.007084   
1971-12-31  0.011478  0.005660  0.004219  0.006710 

In [40]:
models = {
    'PCA - first 8 PCs': PCAFirst8PCsWithCPModel(xr_full=xr, cp_cols=target_cols, components=8),
    #'PCA as in Ludvigson and Ng (2009)': LudvigsonNgWithCPModel(xr_full=xr[target_cols], n_factors=8),
}

In [41]:
results = []
forecast_store = {}

In [42]:
for seed in seeds:
    print(f'Running seed {seed}...')

    for model_name, model in models.items():
        print(f'\nModel: {model_name}')

        for target_name, y in targets.items():
            print(f'  Target: {target_name}')

            y_forecast = wu.expanding_window(
                model,
                X,
                y,
                dates,
                OOS_start,
                gap=0,
                refit_freq=1,
                coef_callback=None
            )

            # store results
            forecast_store.setdefault(target_name, {})
            forecast_store[target_name][model_name] = y_forecast

            # compute metrics
            r2_hist = wu.oos_r2(y, y_forecast, benchmark='hist_mean')
            pval_hist = bu.RSZ_Signif(y, y_forecast)

            results.append({
                'seed': seed,
                'target': target_name,
                'model': model_name,
                'r2_hist_mean': r2_hist,
                'pval_hist_mean': pval_hist
            })

Running seed 1...

Model: PCA - first 8 PCs
  Target: 24


/Users/trineberntsensaether/Documents/Master thesis/master_code/TIO4900-Replication/utils/base_utils.py:465: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  return 1-tstat.cdf(results.tvalues[0], results.nobs-1)


  Target: 36


/Users/trineberntsensaether/Documents/Master thesis/master_code/TIO4900-Replication/utils/base_utils.py:465: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  return 1-tstat.cdf(results.tvalues[0], results.nobs-1)


  Target: 48


...

/Users/trineberntsensaether/Documents/Master thesis/master_code/TIO4900-Replication/utils/base_utils.py:465: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  return 1-tstat.cdf(results.tvalues[0], results.nobs-1)


  Target: 84


/Users/trineberntsensaether/Documents/Master thesis/master_code/TIO4900-Replication/utils/base_utils.py:465: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  return 1-tstat.cdf(results.tvalues[0], results.nobs-1)


  Target: 120


/Users/trineberntsensaether/Documents/Master thesis/master_code/TIO4900-Replication/utils/base_utils.py:465: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  return 1-tstat.cdf(results.tvalues[0], results.nobs-1)


In [43]:
print(X.index[:5])
print(xr.index[:5])

DatetimeIndex(['1971-08-31', '1971-09-30', '1971-10-31', '1971-11-30',
               '1971-12-31'],
              dtype='datetime64[ns]', name='date', freq=None)
DatetimeIndex(['1971-08-31', '1971-09-30', '1971-10-31', '1971-11-30',
               '1971-12-31'],
              dtype='datetime64[ns]', name='date', freq=None)


In [44]:
results_df = pd.DataFrame(results)

results_df_display = results_df.copy()

results_df_display['pval_hist_mean'] = results_df_display.apply(
    lambda row: row['pval_hist_mean'] if row['r2_hist_mean'] > 0 else np.nan,
    axis=1
)

results_df_display['r2_hist_mean'] = results_df_display['r2_hist_mean'] * 100

results_df_display = results_df_display.round({
    'r2_hist_mean': 1,
    'pval_hist_mean': 4
})

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
print(results_df_display)

   seed target              model  r2_hist_mean  pval_hist_mean
0     1     24  PCA - first 8 PCs          -3.3             NaN
1     1     36  PCA - first 8 PCs           2.2          0.0062
2     1     48  PCA - first 8 PCs           4.0          0.0043
3     1     60  PCA - first 8 PCs           5.2          0.0032
4     1     84  PCA - first 8 PCs           5.7          0.0026
5     1    120  PCA - first 8 PCs           7.6          0.0013


In [45]:
for col in target_cols:
    assert xr[col].index.equals(xr[target_cols].index)

In [46]:
for col in target_cols:
    assert targets[col].index.equals(xr[target_cols].index)